In [33]:
import os
import sys
import subprocess

In [34]:
dir_datas = "/Users/latterday/Desktop/Project/proGAT/datas"
file_temp = "/Users/latterday/Desktop/Project/proGAT/file_temp"

In [35]:
os.makedirs(file_temp, exist_ok=True)

In [36]:
sample_id = [
    f for f in os.listdir(dir_datas)
    if f.endswith((".fastq.gz", ".fq.gz", ".fastq", ".fq"))
]

In [ ]:
temp_hifyasm = os.path.join(file_temp, "hifiasm")
os.makedirs(temp_hifyasm, exist_ok=True)

temp_fastplong = os.path.join(file_temp, "fastplong_filtered")

docker_image_hifiasm = "staphb/hifiasm:latest"


for sample in sample_id:
    sample_name = sample

    for suffix in (".fastq.gz", ".fq.gz", ".fastq", ".fq"):
        if sample_name.endswith(suffix):
            sample_name = sample_name.removesuffix(suffix)
            break

    sample_output_dir = os.path.join(
        temp_hifyasm,
        sample_name
    )

    os.makedirs(
        sample_output_dir,
        exist_ok=True
    )

    input_file = os.path.join(
        temp_fastplong,
        sample_name,
        f"{sample_name}_filtered.fastq.gz"
    )

    if not os.path.isfile(input_file):
        raise FileNotFoundError(
            f"Input file not found: {input_file}"
        )

    # hifiasm ONT-only assembly
    subprocess.run(
        [
            "docker", "run", "--rm",
            "--platform", "linux/amd64",
            "-v", f"{input_file}:/input.fastq.gz:ro",
            "-v", f"{sample_output_dir}:/output",
            docker_image_hifiasm,
            "hifiasm",
            "-t", "4",
            "--ont",
            "-o",
            "/output/ONT.asm",
            "/input.fastq.gz"
        ],check=True)